In [1]:
import mesa_reader as mr
import numpy as np
import os
from matplotlib import pyplot as plt, colors
%matplotlib widget

In [2]:
global_dir = '/homes/hanasoge/nipun_mesa/'
global_dir = '/home/nipunghanghas/nipun_mesa/'
# os.chdir(global_dir+'1M_pre_ms_to_wd')
os.chdir(global_dir+'rgb_grid')


In [4]:
# make a MesaData object from a history file
log_dir = './LOGS_to_end_core_h_burn/'
log_dir = './LOGS_to_start_he_core_flash/'
# log_dir = './LOGS'
h = mr.MesaData(log_dir+'history.data')
l = mr.MesaLogDir(log_dir)

In [5]:
tams_age = h.star_age[h.center_h1<1e-4][0]
select_model_number = h.model_number[(h.nu_max > 100)&(h.nu_max < 270)&(h.star_age>tams_age)&(h.delta_Pg>40)&(h.delta_Pg<500)]
req_model_numbers = np.intersect1d(l.model_numbers ,select_model_number)

(array([6, 7]), array([500, 600]))

In [6]:
l.profile_with_model_number(500)

6

In [10]:
if req_model_numbers.shape[0]>0:
    for number in req_model_numbers:
        number_profile = l.profile_with_model_number(number)
        p = l.profile_data(model_number=number)
        if p.model_number!=h.data_at_model_number('model_number',number):
            print(p.model_number,h.data_at_model_number('model_number',number))
            raise Exception('model_number in profile.data does not match with history.data')
        print(p.model_number,h.data_at_model_number('nu_max',number),p.star_age)
        min_freq = h.data_at_model_number('nu_max',number)-h.data_at_model_number('delta_nu',number)
        max_freq = h.data_at_model_number('nu_max',number)+h.data_at_model_number('delta_nu',number)
        write_gyre_file(number_profile,min_freq,max_freq)
else:
    print('No model satisfies the selection criteria.')

500 218.15259426019543 12007152104.986586
600 119.8842599215892 12094343054.225163


sh: 1: /bin/gyre: not found
sh: 1: /bin/gyre: not found


In [8]:
p.bulk_names

('zone',
 'mass',
 'logR',
 'logT',
 'logRho',
 'logP',
 'x_mass_fraction_H',
 'y_mass_fraction_He',
 'z_mass_fraction_metals',
 'pp',
 'cno',
 'tri_alpha')

In [9]:
def write_gyre_file(profile_num,min_freq,max_freq):
    f = open("./work/gyre.in","w")
    f.write("""&constants
/

&model
  model_type = 'EVOL'  ! Obtain stellar structure from an evolutionary model
  file = '/home/nipunghanghas/nipun_mesa/1M_pre_ms_to_wd/LOGS/profile%d.data.GYRE'    ! File name of the evolutionary model
  file_format = 'MESA' ! File format of the evolutionary model
/

&mode
  l = 1 ! Harmonic degree
/

&osc
  outer_bound = 'VACUUM' ! Assume the density vanishes at the stellar surface
/

&rot
/

&num
  diff_scheme = 'COLLOC_GL4' ! 4th-order collocation scheme for difference equations
/

&scan
  grid_type = 'LINEAR' ! Scan grid uniform in inverse frequency
  freq_min = %.3f        ! Minimum frequency to scan from
  freq_max = %.3f        ! Maximum frequency to scan to
  n_freq = 100          ! Number of frequency points in scan
  freq_units = 'UHZ'                   	      ! Units of freq output items
/

&grid
  w_osc = 10 ! Oscillatory region weight parameter
  w_exp = 2  ! Exponential region weight parameter
  w_ctr = 10 ! Central region weight parameter
/

"""%(profile_num,min_freq,max_freq))
    f.write("""
&ad_output
  summary_file_format = 'TXT'
  detail_file_format = 'TXT'
  summary_file = 'summary.txt'                         ! File name for summary file
  summary_item_list = 'l,n_pg,n_p,n_g,freq,freq_units,E_norm' ! Items to appear in summary file
  detail_template = 'profile{}.detail.l%l.n%n.txt'        	      ! File name template for detail files
  detail_item_list = 'l,n_pg,n_p,freq,freq_units,x,xi_r,
                      xi_h,c_1,As,V_2,Gamma_1' 	      ! Items to appear in detail files
  freq_units = 'UHZ'                   	      ! Units of freq output items
/

&nad_output
/

""".format(profile_num))
    f.close()
    cmd = "$GYRE_DIR/bin/gyre ./work/gyre.in"
    k = os.system(cmd)
    return k

In [103]:
os.system("which source")

256

In [86]:
# os.system("bash ~/.mesaenv.sh")
os.system("echo $GYRE_DIR")

0

In [ ]:
&constants
/

&model
  model_type = 'EVOL'  ! Obtain stellar structure from an evolutionary model
  file = '/home/nipunghanghas/nipun_mesa/1M_pre_ms_to_wd/LOGS/profile10.data.GYRE'    ! File name of the evolutionary model
  file_format = 'MESA' ! File format of the evolutionary model
/

&mode
  l = 0 ! Harmonic degree
/

&mode
  l = 1 ! Harmonic degree
/

&mode
  l = 2 ! Harmonic degree
/

&osc
  outer_bound = 'VACUUM' ! Assume the density vanishes at the stellar surface
/

&rot
/

&num
  diff_scheme = 'COLLOC_GL4' ! 4th-order collocation scheme for difference equations
/

&scan
  grid_type = 'LINEAR' ! Scan grid uniform in inverse frequency
  freq_min = 50        ! Minimum frequency to scan from
  freq_max = 300        ! Maximum frequency to scan to
  n_freq = 100          ! Number of frequency points in scan
  freq_units = 'UHZ'                   	      ! Units of freq output items
/

&grid
  w_osc = 10 ! Oscillatory region weight parameter
  w_exp = 2  ! Exponential region weight parameter
  w_ctr = 10 ! Central region weight parameter
/


&ad_output
  summary_file_format = 'TXT'
  detail_file_format = 'TXT'
  summary_file = 'summary.txt'                         ! File name for summary file
  summary_item_list = 'l,n_pg,n_p,n_g,freq,freq_units,E_norm' ! Items to appear in summary file
  detail_template = 'detail.l%l.n%n.txt'        	      ! File name template for detail files
  detail_item_list = 'l,n_pg,n_p,freq,freq_units,x,xi_r,
                      xi_h,c_1,As,V_2,Gamma_1' 	      ! Items to appear in detail files
  freq_units = 'UHZ'                   	      ! Units of freq output items
/

&nad_output
/
